In [ ]:
import numpy as np
import pandas as pd
import time
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolTransforms
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, pearsonr
from openpyxl.styles import Font, Alignment

warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_FOLDER = "developmental toxicity_structuralfeature"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
EXCEL_OUTPUT = os.path.join(OUTPUT_FOLDER, "ANOVA_Significance_Results.xlsx")

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["axes.linewidth"] = 1.0
plt.rcParams["figure.dpi"] = 300
sns.set_palette("Set2")

DIM_F1 = 20
DIM_F2_RAW = 1024
DIM_F3 = 10
DIM_F4 = 43
DIM_HB = 2
DIM_INTER = 14
ASSAY_A_COLS = ["SMILES1-assay1","SMILES1-assay2","SMILES1-assay3"]
ASSAY_B_COLS = ["SMILES2-assay1","SMILES2-assay2","SMILES2-assay3"]
NUM_CLASSES = 5

label_map = {
    "Antagonistic effect":0,
    "Synergistic effect":1,
    "Not mentioned":2,
    "Additive effect":3,
    "Independent effect":4
}
legend_label_dict = {
    0: "Antagonistic",
    1: "Synergistic",
    2: "No documented interaction",
    3: "Additive",
    4: "Independent"
}

def print_progress(stage_name, current, total, start_time):
    elapsed = time.time() - start_time
    eta = 0 if current == 0 else (elapsed / current) * (total - current)
    print(f"【{stage_name}】进度: {current}/{total} | 已耗时:{elapsed:.1f}s | 预估剩余:{eta:.1f}s | 单条平均:{(elapsed/current):.2f}s")

def to_fixed_len(arr, target_len):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    return np.concatenate([arr, np.zeros(target_len-len(arr), dtype=np.float32)]) if len(arr)<target_len else arr[:target_len]

def generate_3d_conformer(mol, max_attempts=10):
    try:
        mol3d = Chem.AddHs(mol)
        for _ in range(max_attempts):
            if AllChem.EmbedMolecule(mol3d, randomSeed=42)==0:
                AllChem.MMFFOptimizeMolecule(mol3d)
                return mol3d
        return None
    except:
        return None

def get_3d_geometric_features(mol3d):
    feats = []
    if mol3d is not None and mol3d.GetNumConformers()>0:
        try:
            conf = mol3d.GetConformer()
            n_atom = mol3d.GetNumAtoms()
            coords = np.array([[conf.GetAtomPosition(i).x,conf.GetAtomPosition(i).y,conf.GetAtomPosition(i).z] for i in range(n_atom)])
            feats.extend(coords.mean(axis=0).tolist()+coords.std(axis=0).tolist()+coords.max(axis=0).tolist()+coords.min(axis=0).tolist())
            bonds = [rdMolTransforms.GetBondLength(conf,b.GetBeginAtomIdx(),b.GetEndAtomIdx()) for b in mol3d.GetBonds()]
            feats += [np.mean(bonds),np.std(bonds),np.max(bonds),np.min(bonds)] if bonds else [0]*4
            angles = []
            for a in range(n_atom):
                for n in mol3d.GetAtomWithIdx(a).GetNeighbors():
                    angles.append(rdMolTransforms.GetAngleDeg(conf,a,n.GetIdx(),a))
            feats += [np.mean(angles),np.std(angles),np.max(angles),np.min(angles)] if angles else [0]*4
            dihed = []
            for b in mol3d.GetBonds():
                a1,a2 = b.GetBeginAtomIdx(),b.GetEndAtomIdx()
                for n1 in mol3d.GetAtomWithIdx(a1).GetNeighbors():
                    for n2 in mol3d.GetAtomWithIdx(a2).GetNeighbors():
                        if n1.GetIdx()!=a2 and n2.GetIdx()!=a1:
                            dihed.append(rdMolTransforms.GetDihedralDeg(conf,n1.GetIdx(),a1,a2,n2.GetIdx()))
            feats += dihed[:20]
        except:pass
    return to_fixed_len(feats, DIM_F4)
    
def extract_mol_features_raw(smi):
    f1,f2,f3,f4,hb = np.zeros(DIM_F1),np.zeros(DIM_F2_RAW),np.zeros(DIM_F3),np.zeros(DIM_F4),np.zeros(DIM_HB)
    if pd.isna(smi) or not isinstance(smi,str) or len(smi.strip())==0:
        return f1,f2,f3,f4,hb
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return f1,f2,f3,f4,hb

    max_pc = Descriptors.MaxPartialCharge(mol)
    min_pc = Descriptors.MinPartialCharge(mol)
    max_pc = max_pc if max_pc is not None else -999.0
    min_pc = min_pc if min_pc is not None else -999.0

    f1 = np.array([
        Descriptors.MolWt(mol),Descriptors.MolLogP(mol),Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),Descriptors.NumHAcceptors(mol),Descriptors.NumValenceElectrons(mol),
        Descriptors.HeavyAtomCount(mol),Descriptors.NHOHCount(mol),Descriptors.NOCount(mol),Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol),Descriptors.NumAliphaticRings(mol),Descriptors.NumSaturatedRings(mol),
        Descriptors.NumHeteroatoms(mol),Descriptors.NumRotatableBonds(mol),
        max_pc, min_pc, Descriptors.MolMR(mol),0,0
    ], dtype=np.float32)
    f1 = to_fixed_len(f1,DIM_F1)

    try:
        f2 = np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=DIM_F2_RAW), dtype=np.float32)
    except:pass
    f3 = np.array([
        Descriptors.MolWt(mol)*0.1,Descriptors.MolLogP(mol)*0.5,Descriptors.TPSA(mol)*0.01,
        Descriptors.NumHDonors(mol),Descriptors.NumHAcceptors(mol),
        Descriptors.NumAromaticRings(mol),Descriptors.NumAliphaticRings(mol),
        Descriptors.NumHeteroatoms(mol),Descriptors.NumRotatableBonds(mol),
        Descriptors.LabuteASA(mol) if Descriptors.LabuteASA(mol) else 0
    ], dtype=np.float32)
    f3 = to_fixed_len(f3,DIM_F3)
    f4 = get_3d_geometric_features(generate_3d_conformer(mol))
    hb = np.array([Descriptors.NumHDonors(mol),Descriptors.NumHAcceptors(mol)], dtype=np.float32)
    return f1,f2,f3,f4,hb

def calc_mol_mixed_interaction(f1a,f1b,f3a,f3b,f4a,f4b,hb_a,hb_b):
    def cos_sim(u,v):
        n1,n2 = np.linalg.norm(u),np.linalg.norm(v)
        return 0 if n1<1e-8 or n2<1e-8 else np.dot(u,v)/(n1*n2)
    def l2_dist(u,v): return np.linalg.norm(u-v)
    cos1,dist1,had1 = cos_sim(f1a,f1b),l2_dist(f1a,f1b),(f1a*f1b).mean()
    cos4,dist4,had4 = cos_sim(f4a,f4b),l2_dist(f4a,f4b),(f4a*f4b).mean()
    had3 = (f3a*f3b).mean()
    da,aa,db,ab = hb_a[0],hb_a[1],hb_b[0],hb_b[1]
    inter = np.array([cos1,dist1,had1,cos4,dist4,had4,had3,da+db,aa+ab,abs(da-db),abs(aa-ab),da*db,aa*ab,0.0], dtype=np.float32)
    return to_fixed_len(inter,DIM_INTER)

def anova_test(feat_mat, label_vec, feat_names):
    start = time.time()
    res = []
    skip_list = []
    total_feat = len(feat_names)
    for idx,name in enumerate(feat_names):
        col = feat_mat[:, idx]
        if np.var(col) < 1e-8:
            skip_list.append(name)
            res.append({"Feature":name,"F_stat":"Skip","p_val":"Skip","Sig":"ns"})
            print_progress("ANOVA Test", idx+1, total_feat, start)
            continue
        groups = [col[label_vec==c] for c in range(NUM_CLASSES)]
        try:
            f_stat,p_val = f_oneway(*groups)
            sig = "***" if p_val<0.001 else "**" if p_val<0.01 else "*" if p_val<0.05 else "ns"
            res.append({"Feature":name,"F_stat":round(f_stat,4),"p_val":round(p_val,6),"Sig":sig})
        except:
            res.append({"Feature":name,"F_stat":"Error","p_val":"Error","Sig":"ns"})
        print_progress("ANOVA Test", idx+1, total_feat, start)
    cost = time.time() - start
    if skip_list:
        print(f"[Warning] Features with near-zero intra-group variance (skipped): {skip_list}")
    df_res = pd.DataFrame(res)
    return df_res

def format_excel_sheet(ws):
    header_font = Font(bold=True)
    center_align = Alignment(horizontal="center", vertical="center")
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = center_align
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 20)
        ws.column_dimensions[column].width = adjusted_width

def draw_box(df, ycol, title, save_name):
    save_path = os.path.join(OUTPUT_FOLDER, save_name)
    fig, ax = plt.subplots(figsize=(11,6))
    sns.boxplot(x="label_name", y=ycol, data=df, width=0.6, linewidth=1.0, fliersize=3, ax=ax)
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def batch_box(df, name_list, prefix):
    start = time.time()
    total = len(name_list)
    for i,name in enumerate(name_list):
        save_name = f"box_{prefix}_{name}.png"
        fig_title = f"Distribution of {name} across five compound interaction categories"
        draw_box(df,name,fig_title,save_name)
        print_progress(f"Plot-{prefix}", i+1, total, start)
    cost = time.time() - start

def batch_box_inter(df_inter, inter_name_list):
    start = time.time()
    total = len(inter_name_list)
    for idx, feat_name in enumerate(inter_name_list):
        save_name = f"box_inter_{feat_name}.png"
        if feat_name == "PaddingZero":
            fig_title = f"Distribution of {feat_name} (zero padding dimension, reference only)"
        else:
            fig_title = f"Pairwise interaction descriptor: {feat_name} across five compound interaction categories"
        draw_box(df_inter, feat_name, fig_title, save_name)
        print_progress("Plot-InteractionAll", idx+1, total, start)
    cost = time.time() - start

def plot_assay_corr(x_arr, label_arr, label_name_arr, assay_idx):
    r, p = pearsonr(x_arr, label_arr)
    fig, ax = plt.subplots(figsize=(10,5))
    df_plot = pd.DataFrame({
        f"Assay{assay_idx}_Avg": x_arr,
        "Interaction_Code": label_arr,
        "Interaction_Type": label_name_arr
    })
    df_plot["Interaction_Label"] = df_plot["Interaction_Code"].map(legend_label_dict)
    sns.scatterplot(data=df_plot, x=f"Assay{assay_idx}_Avg", y="Interaction_Code", hue="Interaction_Label", alpha=0.7, s=20, ax=ax)
    ax.set_title(f"Mean Assay {assay_idx} Toxicity vs Compound Interaction Type (r = {r:.4f}, p = {p:.4e})")
    ax.set_xlabel(f"Averaged toxicity probability of Assay {assay_idx} (Compound A & B)")
    ax.set_ylabel("Compound interaction category code")
    plt.legend(bbox_to_anchor=(1.01,1), loc="upper left")
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_FOLDER, f"scatter_assay{assay_idx}_corr.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return r, p

if __name__ == "__main__":
    total_start = time.time()
    print("="*60)
    print(f"Program initialized, all outputs will be saved to: ./{OUTPUT_FOLDER}/")
    print("="*60)

    t1 = time.time()
    df = pd.read_excel("developmental-toxicity-traindata-Enumerator.xlsx",sheet_name=0) #neurotoxicity/immunotoxicity/hepatotoxicity/reproduction-toxicity-traindata-Enumerator.xlsx
    df["Interaction types of compounds"] = df["Interaction types of compounds"].str.strip()
    df = df[df["Interaction types of compounds"].isin(label_map.keys())].reset_index(drop=True)
    df["label"] = df["Interaction types of compounds"].map(label_map)
    df["label_name"] = df["Interaction types of compounds"]
    print(f"Dataset loaded, total samples: {len(df)}, time cost: {time.time()-t1:.2f}s")

    assay1_A = df["SMILES1-assay1"].values.astype(np.float32)
    assay1_B = df["SMILES2-assay1"].values.astype(np.float32)
    assay1_mean = (assay1_A + assay1_B) / 2
    assay2_A = df["SMILES1-assay2"].values.astype(np.float32)
    assay2_B = df["SMILES2-assay2"].values.astype(np.float32)
    assay2_mean = (assay2_A + assay2_B) / 2
    assay3_A = df["SMILES1-assay3"].values.astype(np.float32)
    assay3_B = df["SMILES2-assay3"].values.astype(np.float32)
    assay3_mean = (assay3_A + assay3_B) / 2
    all_assay_vals = df[ASSAY_A_COLS + ASSAY_B_COLS].values.astype(np.float32)
    all_assay_mean = all_assay_vals.mean(axis=1)

    print("\n==== Step 1: Extract molecular descriptors & pairwise interaction features ====")
    inter_list = []
    f1A,f3A,f4A,hbA,label,label_name = [],[],[],[],[],[]
    sample_total = len(df)
    extract_start = time.time()
    for idx,row in df.iterrows():
        f1a,f2a,f3a,f4a,hba = extract_mol_features_raw(row["SMILES1"])
        f1b,f2b,f3b,f4b,hbb = extract_mol_features_raw(row["SMILES2"])
        inter_list.append(calc_mol_mixed_interaction(f1a,f1b,f3a,f3b,f4a,f4b,hba,hbb))
        f1A.append(f1a)
        f3A.append(f3a)
        f4A.append(f4a)
        hbA.append(hba)
        label.append(row["label"])
        label_name.append(row["label_name"])
        if (idx+1) % 20 == 0 or (idx+1) == sample_total:
            print_progress("Descriptor Extraction", idx+1, sample_total, extract_start)
    inter_mat = np.array(inter_list)
    f1A_mat,f3A_mat,f4A_mat,hbA_mat = np.array(f1A),np.array(f3A),np.array(f4A),np.array(hbA)
    label_arr,label_name_arr = np.array(label),np.array(label_name)
    print(f"All molecular descriptors extracted, total time cost: {time.time()-extract_start:.2f}s")

    f1_names = ["MolWt","LogP","TPSA","HDonor","HAcceptor","ValenceElectron","HeavyAtom","NHOHCount","NOCount","FracCSP3","AromaticRing","AliphaticRing","SatRing","HeteroAtom","RotBond","MaxPartialCharge","MinPartialCharge","MolMR"]
    f3_names = ["Scaled_MolWt","Scaled_LogP","Scaled_TPSA","HDonor","HAcceptor","AromaticRing","AliphaticRing","HeteroAtom","RotBond","LabuteASA"]
    f4_names = ["X_mean","Y_mean","Z_mean","X_std","Y_std","Z_std","X_max","Y_max","Z_max","X_min","Y_min","Z_min","BondLen_mean","BondLen_std","BondLen_max","BondLen_min"]
    hb_names = ["HDonor_num","HAcceptor_num"]
    inter_names = ["F1_CosineSim","F1_L2Dist","F1_Hadamard","F4_CosineSim","F4_L2Dist","F4_Hadamard","F3_Hadamard","HB_SumDonor","HB_SumAcceptor","HB_DiffDonor","HB_DiffAcceptor","HB_MulDonor","HB_MulAcceptor","PaddingZero"]

    df_f1 = pd.DataFrame(f1A_mat[:,:len(f1_names)], columns=f1_names)
    df_f1["label_name"] = label_name_arr
    df_f3 = pd.DataFrame(f3A_mat, columns=f3_names)
    df_f3["label_name"] = label_name_arr
    df_f4 = pd.DataFrame(f4A_mat[:,:len(f4_names)], columns=f4_names)
    df_f4["label_name"] = label_name_arr
    df_hb = pd.DataFrame(hbA_mat, columns=hb_names)
    df_hb["label_name"] = label_name_arr
    df_inter = pd.DataFrame(inter_mat,columns=inter_names)
    df_inter["label_name"] = label_name_arr

    f1_key = f1A_mat[:,:len(f1_names)]
    f4_key = f4A_mat[:,:len(f4_names)]
    all_phys_mat = np.concatenate([f1_key,f4_key,hbA_mat],axis=1)
    all_phys_names = f1_names + f4_names + hb_names

    print("\n" + "="*50)
    print("==== Step 2: Statistical analysis of single-molecule physicochemical descriptors ====")
    print("="*50)
    phys_anova_df = anova_test(all_phys_mat, label_arr, all_phys_names)
    batch_box(df_f1,f1_names,"F1")
    batch_box(df_f3,f3_names,"F3")
    batch_box(df_f4,f4_names,"F4")
    batch_box(df_hb,hb_names,"HB")

    print("\n" + "="*50)
    print("==== Step 3: Correlation analysis between individual assay toxicity and interaction category ====")
    print("="*50)
    corr_start = time.time()
    r1,p1 = plot_assay_corr(assay1_mean, label_arr, label_name_arr, 1)
    print(f"Assay 1 (A+B averaged): Pearson r = {r1:.4f}, p-value = {p1:.6f}")
    r2,p2 = plot_assay_corr(assay2_mean, label_arr, label_name_arr, 2)
    print(f"Assay 2 (A+B averaged): Pearson r = {r2:.4f}, p-value = {p2:.6f}")
    r3,p3 = plot_assay_corr(assay3_mean, label_arr, label_name_arr, 3)
    print(f"Assay 3 (A+B averaged): Pearson r = {r3:.4f}, p-value = {p3:.6f}")

    df_all = pd.DataFrame({"All_Assay_Avg":all_assay_mean,"Interaction_Code":label_arr,"Interaction_Type":label_name_arr})
    df_all["Interaction_Label"] = df_all["Interaction_Code"].map(legend_label_dict)
    fig, ax = plt.subplots(figsize=(10,5))
    sns.scatterplot(data=df_all, x="All_Assay_Avg", y="Interaction_Code", hue="Interaction_Label", alpha=0.7, s=20, ax=ax)
    r_all,p_all = pearsonr(all_assay_mean, label_arr)
    ax.set_title(f"Averaged Toxicity of Six Assays vs Compound Interaction Type (r = {r_all:.4f}, p = {p_all:.4e})")
    ax.set_xlabel("Mean toxicity probability across six in vitro assays (Compound A & B)")
    ax.set_ylabel("Compound interaction category code")
    plt.legend(bbox_to_anchor=(1.01,1), loc="upper left")
    plt.tight_layout()
    scatter_save = os.path.join(OUTPUT_FOLDER, "scatter_all_assay_corr.png")
    plt.savefig(scatter_save, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Six-assay overall average: Pearson r = {r_all:.4f}, p-value = {p_all:.6f}")
    print(f"All assay correlation scatterplots saved, total time cost: {time.time()-corr_start:.2f}s")

    print("\n" + "="*50)
    print("==== Step 4: Statistical analysis of all 14 pairwise molecular interaction descriptors ====")
    print("="*50)
    inter_anova_df = anova_test(inter_mat,label_arr,inter_names)
    batch_box_inter(df_inter, inter_names)

    draw_box(df_inter,"F1_CosineSim", "Cosine similarity of F1 physicochemical descriptors between paired molecules", "box_inter_key_F1_CosineSim.png")
    draw_box(df_inter,"HB_DiffDonor", "Absolute difference of hydrogen bond donors between paired molecules", "box_inter_key_HB_DiffDonor.png")
    print("Key pairwise interaction figures (main manuscript figures) exported separately.")

    print("\n" + "="*50)
    print("==== Step 5: Export ANOVA significance results to Excel ====")
    print("="*50)
    excel_start = time.time()
    with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
        phys_anova_df.to_excel(writer, sheet_name="Single_Molecule_Descriptors", index=False)
        inter_anova_df.to_excel(writer, sheet_name="Pairwise_Interaction_Descriptors", index=False)
    from openpyxl import load_workbook
    wb = load_workbook(EXCEL_OUTPUT)
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        format_excel_sheet(ws)
    wb.save(EXCEL_OUTPUT)
    print(f"ANOVA table successfully saved to: {EXCEL_OUTPUT}")
    print(f"Excel export finished, time cost: {time.time()-excel_start:.2f}s")

    total_cost = time.time() - total_start
    print("\n" + "="*60)
    print(f"All computational pipelines completed! Total runtime: {total_cost:.2f} seconds")
    print(f"✅ All figures & ANOVA table are stored in: ./{OUTPUT_FOLDER}/")
    print("="*60)